In [1]:
import json
import hashlib
import re
from pathlib import Path
from datetime import datetime
from typing import Optional


# ──────────────────────────────────────────────
# CONFIGURACIÓN
# ──────────────────────────────────────────────
MAX_CHUNK_TOKENS = 800      # Máximo ~tokens por chunk (aprox 1 token ≈ 4 chars en español)
OVERLAP_TOKENS = 100        # Solapamiento entre chunks cuando hay que partir un artículo largo
MAX_CHUNK_CHARS = MAX_CHUNK_TOKENS * 4
OVERLAP_CHARS = OVERLAP_TOKENS * 4

# Metadatos de los documentos que procesamos
DOCUMENT_METADATA = {
    "gdpr": {
        "nombre_corto": "RGPD",
        "nombre_completo": "Reglamento General de Protección de Datos",
        "referencia_legal": "Reglamento (UE) 2016/679",
        "fecha_publicacion": "2016-04-27",
        "fuente": "EUR-Lex",
        "url": "https://eur-lex.europa.eu/eli/reg/2016/679/oj",
        "jurisdiccion": "UE"
    },
    "aiact": {
        "nombre_corto": "AI Act",
        "nombre_completo": "Ley de Inteligencia Artificial",
        "referencia_legal": "Reglamento (UE) 2024/1689",
        "fecha_publicacion": "2024-07-12",
        "fuente": "EUR-Lex",
        "url": "https://eur-lex.europa.eu/eli/reg/2024/1689/oj",
        "jurisdiccion": "UE"
    }
}

# Roles que son "ruido" y no queremos en los chunks
SKIP_ROLES = {"pageHeader", "pageFooter", "pageNumber"}


def parse_paragraphs(analyze_result: dict) -> list[dict]:
    """
    Extrae los párrafos relevantes del JSON de Document Intelligence,
    descartando headers, footers y números de página.
    """
    paragraphs = []
    for i, p in enumerate(analyze_result.get("paragraphs", [])):
        role = p.get("role", "none")
        if role in SKIP_ROLES:
            continue
        
        page = None
        if p.get("boundingRegions"):
            page = p["boundingRegions"][0].get("pageNumber")
        
        paragraphs.append({
            "index": i,
            "role": role,
            "content": p["content"].strip(),
            "page": page
        })
    return paragraphs


def detect_article_boundary(paragraph: dict) -> Optional[dict]:
    """
    Detecta si un párrafo es el inicio de un artículo o capítulo.
    Retorna metadata de la sección si la detecta.
    """
    content = paragraph["content"].strip()
    role = paragraph["role"]
    
    # Patrón: "Artículo X" o "Artículo X Título"
    art_match = re.match(r'^Artículo\s+(\d+)\s*(.*)', content, re.IGNORECASE)
    if art_match:
        return {
            "tipo": "articulo",
            "numero": art_match.group(1),
            "titulo_inline": art_match.group(2).strip() if art_match.group(2) else None
        }
    
    # Patrón: "CAPÍTULO I" / "CAPÍTULO I DISPOSICIONES GENERALES"
    cap_match = re.match(
        r'^CAPÍTULO\s+([IVXLCDM]+)\s*(.*)', 
        content, re.IGNORECASE
    )
    if cap_match:
        return {
            "tipo": "capitulo",
            "numero": cap_match.group(1).strip(),
            "titulo_inline": cap_match.group(2).strip() if cap_match.group(2) else None
        }
    
    # Patrón: "TÍTULO X"
    tit_match = re.match(r'^TÍTULO\s+([IVXLCDM]+)\s*(.*)', content, re.IGNORECASE)
    if tit_match:
        return {
            "tipo": "titulo",
            "numero": tit_match.group(1).strip(),
            "titulo_inline": tit_match.group(2).strip() if tit_match.group(2) else None
        }
    
    # Patrón: Sección heading que es solo un título descriptivo (tras un artículo)
    if role == "sectionHeading" and not art_match and not cap_match:
        return {
            "tipo": "subtitulo",
            "titulo_inline": content
        }
    
    return None


def group_by_articles(paragraphs: list[dict]) -> list[dict]:
    """
    Agrupa párrafos consecutivos bajo su artículo/sección correspondiente.
    
    Lógica:
    - Cuando encontramos un 'Artículo N', abrimos un nuevo grupo
    - Los párrafos de sectionHeading justo después se usan como título
    - Cuando encontramos un 'CAPÍTULO X', actualizamos el contexto del capítulo
    - Todo el contenido hasta el siguiente artículo se agrupa junto
    """
    groups = []
    current_chapter = None
    current_article = None
    current_article_title = None
    current_content_parts = []
    current_pages = set()
    
    # Para considerandos / texto previo a artículos
    preamble_parts = []
    
    def flush_article():
        """Guarda el artículo actual si tiene contenido."""
        nonlocal current_article, current_article_title, current_content_parts, current_pages
        if current_content_parts:
            full_text = "\n\n".join(current_content_parts)
            groups.append({
                "capitulo": current_chapter,
                "articulo": current_article,
                "titulo_articulo": current_article_title,
                "contenido": full_text,
                "paginas": sorted(current_pages)
            })
        current_content_parts = []
        current_pages = set()
    
    for para in paragraphs:
        boundary = detect_article_boundary(para)
        
        if boundary and boundary["tipo"] == "capitulo":
            # Nuevo capítulo: flush artículo actual
            titulo = boundary.get("titulo_inline", "")
            current_chapter = f"Capítulo {boundary['numero']}"
            if titulo:
                current_chapter += f" - {titulo}"
            continue
        
        if boundary and boundary["tipo"] == "titulo":
            current_chapter = f"Título {boundary['numero']}"
            if boundary.get("titulo_inline"):
                current_chapter += f" - {boundary['titulo_inline']}"
            continue
        
        if boundary and boundary["tipo"] == "articulo":
            # Nuevo artículo: guardar el anterior
            flush_article()
            current_article = f"Artículo {boundary['numero']}"
            current_article_title = boundary.get("titulo_inline")
            if para.get("page"):
                current_pages.add(para["page"])
            continue
        
        if boundary and boundary["tipo"] == "subtitulo":
            # Subtítulo descriptivo
            if not current_article:
                # Subtítulo antes de cualquier artículo: complemento del capítulo
                if current_chapter and " - " not in current_chapter:
                    current_chapter += f" - {para['content']}"
                continue
            elif not current_article_title:
                # Primer heading tras un artículo = título del artículo
                current_article_title = para["content"]
            elif not current_content_parts:
                current_article_title = para["content"]
            else:
                # Subtítulo dentro del contenido
                current_content_parts.append(f"**{para['content']}**")
            if para.get("page"):
                current_pages.add(para["page"])
            continue
        
        # Párrafo de contenido normal
        if current_article:
            # Estamos dentro de un artículo
            current_content_parts.append(para["content"])
            if para.get("page"):
                current_pages.add(para["page"])
        else:
            # Estamos en considerandos o preámbulo
            preamble_parts.append(para)
    
    # Flush el último artículo
    flush_article()
    
    # Si hay considerandos, agruparlos como chunk especial
    if preamble_parts:
        # Agrupar considerandos consecutivos
        considerando_texts = []
        for p in preamble_parts:
            considerando_texts.append(p["content"])
        
        if considerando_texts:
            groups.insert(0, {
                "capitulo": None,
                "articulo": "Considerandos / Preámbulo",
                "titulo_articulo": "Texto previo a los artículos",
                "contenido": "\n\n".join(considerando_texts),
                "paginas": sorted({p["page"] for p in preamble_parts if p.get("page")})
            })
    
    return groups


def split_long_chunk(text: str, max_chars: int, overlap_chars: int) -> list[str]:
    """
    Divide un texto largo en sub-chunks respetando límites de párrafo.
    Usa solapamiento para mantener contexto entre chunks.
    """
    if len(text) <= max_chars:
        return [text]
    
    # Dividir por párrafos (doble salto de línea)
    paragraphs = text.split("\n\n")
    chunks = []
    current_chunk = []
    current_len = 0
    
    for para in paragraphs:
        para_len = len(para)
        
        if current_len + para_len > max_chars and current_chunk:
            # Guardar chunk actual
            chunks.append("\n\n".join(current_chunk))
            
            # Calcular solapamiento: incluir los últimos párrafos como contexto
            overlap_parts = []
            overlap_len = 0
            for prev_para in reversed(current_chunk):
                if overlap_len + len(prev_para) <= overlap_chars:
                    overlap_parts.insert(0, prev_para)
                    overlap_len += len(prev_para)
                else:
                    break
            
            current_chunk = overlap_parts
            current_len = overlap_len
        
        current_chunk.append(para)
        current_len += para_len
    
    if current_chunk:
        chunks.append("\n\n".join(current_chunk))
    
    return chunks


def generate_chunk_id(doc_key: str, articulo: str, part: int = 0) -> str:
    """Genera un ID único y determinista para cada chunk."""
    raw = f"{doc_key}|{articulo}|{part}"
    return hashlib.md5(raw.encode()).hexdigest()[:12]


def create_chunks(article_groups: list[dict], doc_key: str) -> list[dict]:
    """
    Genera los chunks finales con metadatos completos,
    dividiendo artículos largos si es necesario.
    """
    doc_meta = DOCUMENT_METADATA.get(doc_key, {})
    chunks = []
    
    # Filtrar grupos con contenido demasiado corto (< 50 chars)
    MIN_CONTENT_CHARS = 50
    article_groups = [g for g in article_groups if len(g["contenido"]) >= MIN_CONTENT_CHARS]
    
    for group in article_groups:
        contenido = group["contenido"]
        
        # Construir el encabezado del chunk para contexto
        header_parts = []
        if doc_meta.get("nombre_corto"):
            header_parts.append(doc_meta["nombre_corto"])
        if group["capitulo"]:
            header_parts.append(group["capitulo"])
        if group["articulo"]:
            header_parts.append(group["articulo"])
        if group["titulo_articulo"]:
            header_parts.append(group["titulo_articulo"])
        
        header = " > ".join(header_parts)
        
        # Dividir si es muy largo
        sub_texts = split_long_chunk(contenido, MAX_CHUNK_CHARS, OVERLAP_CHARS)
        total_parts = len(sub_texts)
        
        for part_idx, sub_text in enumerate(sub_texts):
            # Prepend header para dar contexto al embedding
            chunk_text = f"{header}\n\n{sub_text}" if header else sub_text
            
            chunk = {
                "chunk_id": generate_chunk_id(doc_key, group["articulo"] or "preambulo", part_idx),
                "documento": doc_meta.get("nombre_corto", doc_key),
                "referencia_legal": doc_meta.get("referencia_legal", ""),
                "capitulo": group["capitulo"],
                "articulo": group["articulo"],
                "titulo_articulo": group["titulo_articulo"],
                "contenido": chunk_text,
                "contenido_sin_header": sub_text,
                "paginas": group["paginas"],
                "parte": part_idx + 1 if total_parts > 1 else None,
                "total_partes": total_parts if total_parts > 1 else None,
                "num_caracteres": len(chunk_text),
                "num_tokens_estimados": len(chunk_text) // 4,
                "fecha_publicacion": doc_meta.get("fecha_publicacion"),
                "url_fuente": doc_meta.get("url"),
                "jurisdiccion": doc_meta.get("jurisdiccion"),
                "fecha_procesamiento": datetime.now().isoformat()
            }
            chunks.append(chunk)
    
    return chunks


def process_document(json_path: str, doc_key: str) -> list[dict]:
    """Pipeline completo: JSON → párrafos → grupos por artículo → chunks."""
    
    print(f"\n{'='*60}")
    print(f"Procesando: {doc_key.upper()} ({json_path})")
    print(f"{'='*60}")
    
    # 1. Cargar JSON
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    analyze_result = data.get("analyzeResult", data)
    
    # 2. Extraer párrafos relevantes
    paragraphs = parse_paragraphs(analyze_result)
    print(f"  Párrafos extraídos (sin headers/footers): {len(paragraphs)}")
    
    # 3. Agrupar por artículos
    article_groups = group_by_articles(paragraphs)
    print(f"  Grupos de artículos detectados: {len(article_groups)}")
    
    for g in article_groups:
        label = g["articulo"] or "Preámbulo"
        title = f" - {g['titulo_articulo']}" if g["titulo_articulo"] else ""
        chars = len(g["contenido"])
        print(f"    → {label}{title} ({chars} chars, págs {g['paginas']})")
    
    # 4. Generar chunks
    chunks = create_chunks(article_groups, doc_key)
    print(f"  Chunks generados: {len(chunks)}")
    
    for c in chunks:
        part_info = f" (parte {c['parte']}/{c['total_partes']})" if c['parte'] else ""
        print(f"    → [{c['chunk_id']}] {c['articulo']}{part_info} "
              f"- {c['num_caracteres']} chars (~{c['num_tokens_estimados']} tokens)")
    
    return chunks


def main():
    """Procesa todos los documentos y genera los ficheros de chunks."""
    
    input_dir = Path("../json/base")
    output_dir = Path("../json/chunk")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Mapeo de archivos a claves de documento
    files = {
        "gdpr": input_dir / "gdpr_recortado.json",
        "aiact": input_dir / "aiact_recortado.json"
    }
    
    all_chunks = []
    
    for doc_key, filepath in files.items():
        if filepath.exists():
            chunks = process_document(str(filepath), doc_key)
            all_chunks.extend(chunks)
            
            # Guardar chunks individuales por documento
            output_file = output_dir / f"chunks_{doc_key}.json"
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(chunks, f, ensure_ascii=False, indent=2)
            print(f"\n  ✅ Guardado: {output_file}")
        else:
            print(f"\n  ⚠️ No encontrado: {filepath}")
    
    # Guardar todos los chunks combinados (para indexación)
    combined_file = output_dir / "chunks_all.json"
    with open(combined_file, "w", encoding="utf-8") as f:
        json.dump(all_chunks, f, ensure_ascii=False, indent=2)
    
    print(f"\n{'='*60}")
    print(f"RESUMEN FINAL")
    print(f"{'='*60}")
    print(f"  Total chunks generados: {len(all_chunks)}")
    print(f"  Archivo combinado: {combined_file}")
    
    # Estadísticas
    avg_chars = sum(c["num_caracteres"] for c in all_chunks) / len(all_chunks) if all_chunks else 0
    avg_tokens = sum(c["num_tokens_estimados"] for c in all_chunks) / len(all_chunks) if all_chunks else 0
    print(f"  Media caracteres/chunk: {avg_chars:.0f}")
    print(f"  Media tokens/chunk (estimado): {avg_tokens:.0f}")
    
    return all_chunks


if __name__ == "__main__":
    main()


Procesando: GDPR (..\json\base\gdpr_recortado.json)
  Párrafos extraídos (sin headers/footers): 37
  Grupos de artículos detectados: 5
    → Considerandos / Preámbulo - Texto previo a los artículos (36 chars, págs [1])
    → Artículo 1 - Objeto (606 chars, págs [1])
    → Artículo 2 - Ámbito de aplicación material (1544 chars, págs [1])
    → Artículo 3 - Ámbito territorial (960 chars, págs [1, 2])
    → Artículo 4 - Definiciones (3051 chars, págs [2])
  Chunks generados: 4
    → [cc61996cbe9a] Artículo 1 - 673 chars (~168 tokens)
    → [9fad5bbaa2b3] Artículo 2 - 1634 chars (~408 tokens)
    → [7bcc2d10c46e] Artículo 3 - 1039 chars (~259 tokens)
    → [0c3f17a68580] Artículo 4 - 3124 chars (~781 tokens)

  ✅ Guardado: ..\json\chunk\chunks_gdpr.json

Procesando: AIACT (..\json\base\aiact_recortado.json)
  Párrafos extraídos (sin headers/footers): 32
  Grupos de artículos detectados: 3
    → Considerandos / Preámbulo - Texto previo a los artículos (4016 chars, págs [1])
    → Artículo 